# 00 · Setup and data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheridanLu/detecting-the-detector/blob/main/notebooks/00_setup_and_data.ipynb)

**Goal.** Get the repository and its data, check that the data are exactly the files the original experiment used,
and check that no test essay was ever seen during training.

| Mode | What runs | Expected time | GPU |
|---|---|---|---|
| SMOKE | all checks on a small sample of the texts | < 30 s | no |
| FAST (default) | all checks on all texts | about 1 min | no |
| FULL | same as FAST (nothing heavier to do in this notebook) | about 1 min | no |

Nothing in this notebook trains a model. Use *Runtime > Run all*.

In [1]:
# --- Setup: find the repository (or download it on Colab) and install anything Colab does not have ---
import os, sys, time, pathlib, subprocess
T0 = time.time()
REPO_URL = "https://github.com/SheridanLu/detecting-the-detector"

def _is_repo(p):
    return (p / "ddlib" / "__init__.py").is_file() and (p / "data").is_dir()

def _locate_or_fetch():
    here = pathlib.Path.cwd().resolve()
    for p in [pathlib.Path(os.environ.get("DD_REPO", here)), here, *here.parents]:
        if _is_repo(p):
            return p                                   # mode 1: already inside the repo
    dest = pathlib.Path("/content" if os.path.isdir("/content") else here) / "detecting-the-detector"
    if not _is_repo(dest) and not dest.exists():       # mode 2: git clone (token optional, for a private repo)
        tok = os.environ.get("GH_TOKEN")
        try:
            from google.colab import userdata
            tok = userdata.get("GH_TOKEN") or tok
        except Exception:
            pass
        url = REPO_URL.replace("https://", f"https://x-access-token:{tok}@") if tok else REPO_URL
        subprocess.run(["git", "clone", "--depth", "1", "-q", url, str(dest)], env=dict(os.environ, GIT_TERMINAL_PROMPT="0"))
        if tok and dest.exists():
            subprocess.run(["git", "-C", str(dest), "remote", "set-url", "origin", REPO_URL])   # token not kept on disk
    if _is_repo(dest):
        return dest
    return None                                        # mode 3 (zip upload) is handled by ddlib below

root = _locate_or_fetch()
if root is None:
    # Zip fallback: needs the small ddlib/paths.py helper, which we cannot import yet, so do it here.
    print("Could not find or clone the repository.\n"
          "Fix 1: add a Colab secret GH_TOKEN (key icon, left sidebar) with read access, allow notebook access, re-run.\n"
          f"Fix 2: download the repo as a zip ({REPO_URL} > Code > Download ZIP) and upload it when asked.")
    from google.colab import files
    import zipfile
    up = files.upload()
    zipfile.ZipFile(next(k for k in up if k.endswith(".zip"))).extractall("/content")
    root = next(p.parent.parent for p in pathlib.Path("/content").glob("*/ddlib/__init__.py") if _is_repo(p.parent.parent))
sys.path.insert(0, str(root))
os.chdir(root)                                         # all paths below are relative to the repo root

_req = [l.strip() for l in open("requirements-colab.txt") if l.strip() and not l.startswith("#")]
if _req:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_req], check=True)

from ddlib import get_repo
REPO = get_repo()
print("Python", sys.version.split()[0])

Repository found. Data folder present: True
Python 3.12.16


## Configuration
SMOKE checks only the first 3,000 training-pool texts so that automated tests finish in seconds.
FAST and FULL check every text.

In [2]:
# --- CONFIG ---------------------------------------------------------------------------------------
# MODE = "SMOKE" (seconds; tiny version used for automated testing), "FAST" (default) or "FULL" (paper setting).
# Change the string below to switch. (The DD_MODE environment variable is only used by automated tests.)
MODE = os.environ.get("DD_MODE", "FAST").upper()
assert MODE in ("SMOKE", "FAST", "FULL"), MODE
SAMPLE = 3000 if MODE == "SMOKE" else None   # None = use every text
print("MODE =", MODE)

MODE = FAST


## 1. What is in `data/`
All files the notebooks need are inside the repository; nothing is downloaded from elsewhere.
`data/README.md` lists every file with its columns and origin.

In [3]:
import json, hashlib
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 80)
from ddlib import data as D

rows = []
for f in sorted(pathlib.Path("data").rglob("*")):
    if f.is_file() and f.name != ".DS_Store":
        rows.append(dict(folder=str(f.parent), file=f.name, MB=round(f.stat().st_size / 1e6, 2)))
inv = pd.DataFrame(rows)
print(f"{len(inv)} files, {inv.MB.sum():.0f} MB in total; largest file {inv.MB.max():.0f} MB")
display(inv.groupby("folder").agg(files=("file", "size"), MB=("MB", "sum")).round(1))

122 files, 181 MB in total; largest file 41 MB


,files,MB
folder,,
data,1,0.0
data/commercial,1,0.0
data/followup,2,0.1
data/followup/trainsets,10,106.0
data/preds/binoculars,1,0.3
data/preds/fastdetect,1,0.2
data/preds/lr,40,11.2
data/preds/qwen,8,1.4
data/preds/roberta,40,6.8


## 2. Are these the original files?
When the splits were built on the A40 server, the SHA-256 fingerprint of each file was stored in `split_summary.json`.
If even one byte had changed, the fingerprint below would differ.

In [4]:
summary = D.load_split_summary()
checks = []
for f in ["exams.parquet", "pool.parquet", "val.parquet", "manifests.json"]:
    got = hashlib.sha256(pathlib.Path("data/splits", f).read_bytes()).hexdigest()
    exp = summary[f"sha256_{f}"]
    checks.append(dict(file=f, sha256=got[:16] + "...", matches_original=got == exp))
checks = pd.DataFrame(checks); display(checks)
print("FINGERPRINT CHECK:", "PASS" if checks.matches_original.all() else "FAIL")

,file,sha256,matches_original
0,exams.parquet,6dd0567823d71721...,True
1,pool.parquet,d0fc60f3d4aaa081...,True
2,val.parquet,3e26d7dad195b8ec...,True
3,manifests.json,e5f15e1787cadb20...,True


FINGERPRINT CHECK: PASS


## 3. How big is each part?
* **Test sets (exams E1-E7)** are never used for training. E1 is the Liang et al. (2023) benchmark: 91 real TOEFL essays
  by non-native writers, 88 US 8th-grade essays (Hewlett), 70 college-application and 145 Stanford CS224N essays,
  plus 352 GPT-3 essays. E2-E5 are other non-native / native essay sets; E6 and E7 are AI essays.
* **Pool** = essays the training sets are drawn from. **Validation** = 600 essays used only to pick the 90%-detection threshold.
* **Training sets (arms)**: every arm has 4,000 human + 4,000 AI essays. Only the human side changes.

In [5]:
exams, pool, val = D.load_exams(), D.load_pool(), D.load_val()
manifests = D.load_manifests()
desc = {"E1": "Liang et al. 2023 benchmark", "E2": "PERSUADE same-prompt: ELL vs non-ELL", "E3": "W&I (A/B/C levels) + LOCNESS native",
        "E4": "FCE exam answers", "E5": "ELLIPSE test half", "E6": "AI: daigt Llama + HC-Var ChatGPT", "E7": "AI: ArguGPT on TOEFL prompts"}
ec = exams.groupby(["exam", "corpus"]).size().rename("essays").reset_index()
ec["test set"] = ec.exam.map(desc)
display(ec)
print("Total test essays:", len(exams))
display(pool.groupby(["source", "label"]).size().rename("essays").reset_index().assign(label=lambda d: d.label.map({0: "human", 1: "AI"})))
display(val.groupby(["source", "label"]).size().rename("essays").reset_index().assign(label=lambda d: d.label.map({0: "human", 1: "AI"})))

# recount and compare with the numbers stored when the splits were built
same = (ec.set_index(ec.exam + "|" + ec.corpus).essays.to_dict() == summary["exam_counts"]
        and pool.source.value_counts().to_dict() == summary["pool_counts"]
        and val.source.value_counts().to_dict() == summary["val_counts"])
print("COUNTS MATCH split_summary.json:", "PASS" if same else "FAIL")

,exam,corpus,essays,test set
0,E1,CS224N_gpt3PromptEng_145,145,Liang et al. 2023 benchmark
1,E1,CS224N_gpt3_145,145,Liang et al. 2023 benchmark
2,E1,CS224N_real_145,145,Liang et al. 2023 benchmark
3,E1,CollegeEssay_gpt3PromptEng_31,31,Liang et al. 2023 benchmark
4,E1,CollegeEssay_gpt3_31,31,Liang et al. 2023 benchmark
5,E1,CollegeEssay_real_70,70,Liang et al. 2023 benchmark
6,E1,HewlettStudentEssay_GPTsimplify_88,88,Liang et al. 2023 benchmark
7,E1,HewlettStudentEssay_real_88,88,Liang et al. 2023 benchmark
8,E1,TOEFL_gpt4polished_91,91,Liang et al. 2023 benchmark
9,E1,TOEFL_real_91,91,Liang et al. 2023 benchmark


Total test essays: 13672


,source,label,essays
0,daigt_ai,AI,7708
1,ellipse,human,1008
2,hcvar_ai,AI,3950
3,persuade_ELL,human,1000
4,persuade_nonELL,human,21345


,source,label,essays
0,daigt_ai,AI,150
1,hcvar_ai,AI,150
2,persuade_nonELL,human,300


COUNTS MATCH split_summary.json: PASS


In [6]:
# What each training arm contains (seed 0 shown; other seeds draw different essays in the same proportions)
src = pool.set_index("id").source
names = {"persuade_nonELL": "human: native (PERSUADE)", "ellipse": "human: ELLIPSE (non-native)",
         "persuade_ELL": "human: PERSUADE-ELL (non-native)", "C": "human: FCE + W&I (non-native, exam-style)",
         "D": "human: HC-Var (native, other kinds)", "daigt_ai": "AI", "hcvar_ai": "AI", "ai": "AI", "native": "human: native (PERSUADE)"}
rows = []
for arm in "AB":
    m = manifests[f"seed0_{arm}_n4000"]
    rows.append(dict(arm=arm, **src.loc[m["human"] + m["ai"]].map(names).value_counts().to_dict()))
for arm in "CD":
    t = pd.read_parquet(f"data/followup/trainsets/seed0_{arm}.parquet")
    rows.append(dict(arm=arm, **t.part.map(names).value_counts().to_dict()))
arms = pd.DataFrame(rows).set_index("arm").fillna(0).astype(int)
arms = arms[[c for c in dict.fromkeys(names.values()) if c in arms]]
display(arms)
print("Runs available: arms A/B at 1,000 / 2,000 / 4,000 essays per class x 5 seeds =", len(manifests), "manifests;",
      "arms C/D x 5 seeds =", len(list(pathlib.Path("data/followup/trainsets").glob("*.parquet"))), "saved training sets")

,human: native (PERSUADE),human: ELLIPSE (non-native),human: PERSUADE-ELL (non-native),"human: FCE + W&I (non-native, exam-style)","human: HC-Var (native, other kinds)",AI
arm,,,,,,
A,4000,0,0,0,0,4000
B,2000,1000,1000,0,0,4000
C,2000,0,0,2000,0,4000
D,2000,0,0,0,2000,4000


Runs available: arms A/B at 1,000 / 2,000 / 4,000 essays per class x 5 seeds = 30 manifests; arms C/D x 5 seeds = 10 saved training sets


## 4. Zero-overlap check
If a test essay (or a near copy) was also in the training data, the test would be too easy.
The rule from the original scripts: two essays count as the same if their whole text matches, or their first
200 characters match, after collapsing all whitespace. We check:

1. every pool text that appears in any arm A/B training set vs every test essay;
2. the validation set vs every test essay;
3. the arm C/D training sets vs the test essays that are still used for evaluation (arm C trains on half of FCE and
   W&I; that half is removed from every evaluation, see `data/followup/train_exam_ids.json`).

We run it twice: on the raw text (as the original split builder did) and after `clean()` (stricter, because cleaning
makes small typographic differences disappear). Expected result: 0 everywhere.

In [7]:
from ddlib.overlap import build_index, count_overlap
from ddlib import clean

used_ids = sorted({i for m in manifests.values() for i in m["human"] + m["ai"]})
used = pool.set_index("id").loc[used_ids, "text"]
if SAMPLE: used = used.iloc[:SAMPLE]
tid = D.train_exam_ids()
eval_exams = exams[~exams.essay_id.isin(tid)]
trainsets = pd.concat([pd.read_parquet(f).assign(run=f.stem) for f in sorted(pathlib.Path("data/followup/trainsets").glob("*.parquet"))])
cd_extra = trainsets[trainsets.part.isin(["C", "D"])].text.drop_duplicates()   # the essays that differ from arms A/B
if SAMPLE: cd_extra = cd_extra.iloc[:SAMPLE]

rows = []
for version, f in (("raw text", lambda s: list(s)), ("after clean()", lambda s: [clean(t) for t in s])):
    idx_all = build_index(f(exams.text)); idx_eval = build_index(f(list(eval_exams.text) + list(val.text)))
    for name, texts, idx in (("arm A/B training texts vs all test essays", used, idx_all),
                             ("validation set vs all test essays", val.text, idx_all),
                             ("arm C/D added essays vs evaluated test essays + validation", cd_extra, idx_eval)):
        nf, n2, na = count_overlap(f(texts), idx)
        rows.append(dict(text_version=version, comparison=name, texts_checked=len(texts), same_full_text=nf, same_first_200=n2, overlapping=na))
ov = pd.DataFrame(rows); display(ov)
# The check must be able to fail: plant 5 test essays (with extra spaces) among the training texts and count again.
planted = list(used.iloc[:100]) + ["  " + t.replace(" ", "  ") for t in exams.text.iloc[:5]]
ctrl = count_overlap(planted, build_index(exams.text))[2]
print("Control: 5 planted test essays found by the check ->", ctrl, "(must be 5)")
print("ZERO-OVERLAP CHECK:", "PASS (0 overlapping essays; control found 5/5)" if ov.overlapping.sum() == 0 and ctrl == 5
      else f"FAIL ({ov.overlapping.sum()} overlaps; control found {ctrl}/5)")
print("Archived values from the original run: split builder final_overlap_train_val_vs_exams =",
      summary["final_overlap_train_val_vs_exams"], "| follow-up gate:", json.load(open("data/followup/overlap_gate.json")))

,text_version,comparison,texts_checked,same_full_text,same_first_200,overlapping
0,raw text,arm A/B training texts vs all test essays,25542,0,0,0
1,raw text,validation set vs all test essays,600,0,0,0
2,raw text,arm C/D added essays vs evaluated test essays + validation,10156,0,0,0
3,after clean(),arm A/B training texts vs all test essays,25542,0,0,0
4,after clean(),validation set vs all test essays,600,0,0,0
5,after clean(),arm C/D added essays vs evaluated test essays + validation,10156,0,0,0


Control: 5 planted test essays found by the check -> 5 (must be 5)
ZERO-OVERLAP CHECK: PASS (0 overlapping essays; control found 5/5)
Archived values from the original run: split builder final_overlap_train_val_vs_exams = 0 | follow-up gate: {'exam_style_train_pool': 3501, 'removed_overlap_C': 0, 'hcvar_human_pool': 10000, 'removed_overlap_D': 0, 'final_overlap_C': 0, 'final_overlap_D': 0}


## 5. Format check
An earlier attempt (not used for any result) let RoBERTa learn a shortcut: "no paragraph breaks + curly quotes = AI".
So every text goes through `clean()` (copied unchanged from the original `a40/common.py`): line breaks collapsed,
curly quotes and long dashes made straight, broken Windows-1252 characters repaired, look-alike Cyrillic letters
replaced. The check below counts texts that still contain any of these artefacts. The raw rows show the check can
fail; the cleaned rows must be all zero.

In [8]:
from ddlib import format_gate
pools = {"pool": pool.text, "validation": val.text, "test (exams)": exams.text, "arm C/D added essays (saved already cleaned)": cd_extra}
if SAMPLE: pools = {k: v.iloc[:SAMPLE] for k, v in pools.items()}
_, raw_table = format_gate({k + " (raw)": list(v) for k, v in pools.items()})
ok, clean_table = format_gate({k + " (cleaned)": [clean(t) for t in v] for k, v in pools.items()})
display(pd.concat([raw_table, clean_table]))
print("FORMAT CHECK after clean():", "PASS" if ok else "FAIL")

,n_texts,line break / tab,double space,leading/trailing space,curly quote,long dash / minus,CP-1252 control char,mojibake 'A-circumflex',Cyrillic letter
pool,,,,,,,,,
pool (raw),35011,29845,9109,12624,704,276,204,58,171
validation (raw),600,435,129,170,17,7,3,0,2
test (exams) (raw),13672,11626,4906,5706,1005,118,22,3,10
arm C/D added essays (saved already cleaned) (raw),10156,0,0,0,0,0,0,0,0
pool (cleaned),35011,0,0,0,0,0,0,0,0
validation (cleaned),600,0,0,0,0,0,0,0,0
test (exams) (cleaned),13672,0,0,0,0,0,0,0,0
arm C/D added essays (saved already cleaned) (cleaned),10156,0,0,0,0,0,0,0,0


FORMAT CHECK after clean(): PASS


## 6. A few essays, before and after cleaning
Short excerpts only (first 300 characters). The first two are human essays from the Liang test set; the third is an
AI essay from the training pool; the last is a pool essay where cleaning visibly changes the text.

In [9]:
import textwrap
def show(label, text, n=300):
    print(f"--- {label}\nraw    : {textwrap.shorten(repr(text[:n]), 340)}\ncleaned: {clean(text)[:n]}\n")
ex = exams.set_index("essay_id")
show("TOEFL essay (non-native writer), TOEFL_real_91#0", ex.loc["TOEFL_real_91#0", "text"])
show("US 8th-grade essay (native writer), HewlettStudentEssay_real_88#0", ex.loc["HewlettStudentEssay_real_88#0", "text"])
show("AI essay from the pool (daigt)", pool[pool.source == "daigt_ai"].text.iloc[0])
changed = pool[pool.text.str.contains("[‘’“”]", regex=True) & pool.text.str.contains("\n")].text.iloc[0]
show("pool essay with curly quotes and line breaks", changed)

--- TOEFL essay (non-native writer), TOEFL_real_91#0
raw    : 'I prefer to cook at home. First of all, it is cheaper and safer than the restaurant. Because I can buy the things needed for cooking, and I can make sure that they are clean and fresh. Second, making a good dinner helps me obtain a sense of achievement. On every Spring festival, I always make dinner'
cleaned: I prefer to cook at home. First of all, it is cheaper and safer than the restaurant. Because I can buy the things needed for cooking, and I can make sure that they are clean and fresh. Second, making a good dinner helps me obtain a sense of achievement. On every Spring festival, I always make dinner

--- US 8th-grade essay (native writer), HewlettStudentEssay_real_88#0
raw    : 'Dear editor, I agree with having computers in our homes. I think that computers are good because of these reason. The computer has a-lot of information that you can look up. Also the computer can save a-lot of important things you need. Then yo

In [10]:
print(f"Notebook finished in {time.time() - T0:.0f} s (mode {MODE}).")

Notebook finished in 29 s (mode FAST).
